In [1]:
import pandas as pd
import unicodedata

def remover_acento(texto): #-------- #Define a função, que recebe um valor chamado texto
    if pd.isna(texto): #-------- #Se a célula for vazia (Nula/NaN) ⬇
        return texto #-------- #Retorna o mesmo texto vazio, sem processar
    return unicodedata.normalize('NFKD', str(texto)).encode('ASCII', 'ignore').decode('ASCII'); #Se tiver dado: transforma em texto, separa letras do acento e exclui os acentos
    
df = pd.read_csv('../base_csv/original/valores.csv', encoding='latin1', sep=';') #Lê o CSV e guarda na variável df, latin1 e ; por padrão Excel BR

df.columns = df.columns.str.lower() #Coloca todos os nomes de coluna em minúsculo, evitando erro de case-sensitive no Supabase

df = df.map(remover_acento) #Aplica a função remover_acento em cada célula do df, removendo os acentos de toda a tabela

colunas_data = ['registro', 'alteracao'] #Lista com o nome das colunas que são datas
for coluna in colunas_data: #Percorre cada nome da lista, um de cada vez
    df[coluna] = pd.to_datetime(df[coluna], dayfirst=True).dt.strftime('%Y-%m-%d') #Converte pra data (dia primeiro) e formata como aaaa-mm-dd


for coluna in ['valor custo', 'valor venda']:
    df[coluna] = df[coluna].astype(str).str.replace('R$', '', regex=False)
    df[coluna] = df[coluna].str.strip()
    df[coluna] = df[coluna].str.replace('.', '', regex=False)
    df[coluna] = df[coluna].str.replace(',', '.', regex=False)
    df[coluna] = pd.to_numeric(df[coluna])

df = df.rename(columns={'valor custo': 'valor_custo', 'valor venda': 'valor_venda'}) #Renomeia pra bater com os nomes das colunas no Supabase

df.to_csv('../base_csv/tratado/valores_tratado.csv', index=False, sep=';', encoding='utf-8-sig')
print("Tratamento concluído! Arquivo salvo em base_csv/tratado/")

Tratamento concluído! Arquivo salvo em base_csv/tratado/
